# WeaveIQ: Data Preparation and Grouped Split

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

# Locate the project whether the kernel starts in notebooks or WeaveIQ
project = Path.cwd()
if not (project / "data").is_dir():
    project = project.parent

candidate_path = project / "data/processed/production_candidates.csv"
assert candidate_path.exists(), f"File not found: {candidate_path}"

data = pd.read_csv(
    candidate_path,
    dtype={"ID": "string"},
    low_memory=False
)

target = "Rej_and_cut_Piece"

assert data["ID"].notna().all(), "Missing IDs found."
assert data["ID"].str.strip().ne("").all(), "Empty IDs found."
assert data["source_row"].is_unique, "Repeated source-row identifiers."

assert np.isfinite(data[target]).all(), "Missing or invalid targets."
assert data[target].ge(0).all(), "Negative targets found."

assert np.isfinite(data["original_production"]).all()
assert data["original_production"].gt(0).all()

print("Checks passed.")
print("Records:", len(data))
print("Distinct IDs:", data["ID"].nunique())

Checks passed.
Records: 19969
Distinct IDs: 946


In [2]:
# Reserve approximately 20% of IDs for final testing
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

dev_positions, test_positions = next(
    splitter.split(data, groups=data["ID"])
)

development = data.iloc[dev_positions].copy()
test = data.iloc[test_positions].copy()

assert set(development["ID"]).isdisjoint(set(test["ID"]))

# Assign five grouped validation folds to development records
development["cv_fold"] = -1
group_cv = GroupKFold(n_splits=5)

for fold, (train_positions, validation_positions) in enumerate(
    group_cv.split(development, groups=development["ID"])
):
    train_ids = set(development.iloc[train_positions]["ID"])
    validation_ids = set(development.iloc[validation_positions]["ID"])
    assert train_ids.isdisjoint(validation_ids)

    development.iloc[
        validation_positions,
        development.columns.get_loc("cv_fold")
    ] = fold

# Save data and a record of the split
output = project / "data/processed"
output.mkdir(parents=True, exist_ok=True)

development.to_csv(output / "development.csv", index=False)
test.to_csv(output / "test_holdout.csv", index=False)

manifest = data[["source_row", "ID"]].copy()
manifest["split"] = np.where(
    manifest["ID"].isin(test["ID"]), "test", "development"
)
manifest.to_csv(output / "split_manifest.csv", index=False)

print("Development IDs:", development["ID"].nunique())
print("Test IDs:", test["ID"].nunique())
print("Development rows:", len(development))
print("Test rows:", len(test))
print("ID overlap: 0")
print("Five development folds created.")
print("Files saved successfully.")

Development IDs: 756
Test IDs: 190
Development rows: 16139
Test rows: 3830
ID overlap: 0
Five development folds created.
Files saved successfully.
